# Capítulo 6: Obtendo Dados

**Bases 5 — Ciência de Dados** · notebook de aula

Cada célula de código é a mesma do livro e roda na ordem em que aparece — execute de cima para baixo. Versão publicada deste capítulo: [https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/index.html)

> **Gerado automaticamente a partir dos `.qmd` do livro por `scripts/gerar-notebooks.py`.** Edições feitas aqui se perdem no próximo `make notebooks`; para mudar o conteúdo, edite o `.qmd`.

In [ ]:
# Põe o diretório de trabalho na raiz do projeto — é o que faz
# `from scratch...` e os caminhos `dados/...` funcionarem. No livro isso vem
# do `execute-dir: project` do Quarto; aqui é feito à mão.
#
# No Colab não existe cópia do projeto, então esta célula clona uma. É rápido
# (clone raso) e acontece só na primeira execução da sessão.
import os
import subprocess
import sys

REPO = "https://github.com/BragaD/UnDF-Bases5-CienciaDeDados-202602.git"


def raiz_do_projeto(inicio="."):
    """Sobe os diretórios até achar o `_quarto.yml`. None se não houver."""
    atual = os.path.abspath(inicio)
    while not os.path.exists(os.path.join(atual, "_quarto.yml")):
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai
    return atual


raiz = raiz_do_projeto()
if raiz is None:
    destino = "/content/bases5" if os.path.isdir("/content") else "bases5"
    if not os.path.isdir(destino):
        print("baixando o material da disciplina...")
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    raiz = raiz_do_projeto(destino)

os.chdir(raiz)
if raiz not in sys.path:
    sys.path.insert(0, raiz)

%matplotlib inline
print("diretório de trabalho:", os.getcwd())

> **📌 Nota**
>
> Este capítulo corresponde ao capítulo 9 de Grus (2019).

> Para escrevê-lo, levei três meses; para concebê-lo, três minutos; para reunir os dados nele, a vida inteira.
>
> — F. Scott Fitzgerald

O [Capítulo 1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap01/index.html) entregou os dados prontos: um `dict` de usuários aqui, uma lista de pares de amizade ali, digitados diretamente no código. Foi proposital — o objetivo daquele capítulo era mostrar o formato do trabalho antes das ferramentas —, mas também foi uma ficção confortável. Na vida real ninguém entrega o dado pronto na sua mão, e o próprio Capítulo 1 avisou que essa lacuna seria cobrada: "obter dado é um problema próprio, e o Capítulo 6 é dedicado a ele."

Este é esse capítulo. Ele não ensina um algoritmo — não há um modelo para treinar nem uma métrica para otimizar. As quatro seções seguem uma escada de fragilidade crescente: primeiro o seu próprio disco (seção 1), depois o HTML de outra pessoa (seção 2), depois o contrato formal de outra pessoa — uma API (seção 3) —, e por fim a permissão de outra pessoa para acessá-la (seção 4). Cada degrau depende de menos coisa que você controla, e todo capítulo daqui para frente pressupõe que você sabe descer por ela.

> **🔷 Conceito**
>
> Toda fonte de dado externa — um arquivo, uma página, uma API — é um **contrato que você não escreveu e não controla**. O custo de obter o dado é proporcional a quanto controle você perdeu: um arquivo seu (seção 1) você controla inteiramente; a permissão de terceiros para acessar uma rede social (seção 4) você não controla nada. É esse eixo, não a lista de ferramentas, que organiza o capítulo.

O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html), logo a seguir, parte do princípio de que o dado já está num formato manipulável — uma tabela do `pandas`, listas e `dict`s, e, quando o dado é numérico e vai virar conta, arrays — e ensina a limpá-lo e transformá-lo. Este capítulo é o que garante que o dado chegou até ali.

Consequência prática dessa escada: quanto mais baixo o degrau, mais o exemplo depende de um serviço de terceiros continuar no ar e continuar respondendo do mesmo jeito. Por isso os exemplos que precisariam de uma requisição em tempo real aparecem aqui como ilustração, e o que roda de verdade roda sobre dado salvo ou inventado — o serviço muda, a técnica não.

Ao final deste capítulo, você será capaz de:

- Ler e escrever arquivos de texto e arquivos delimitados (CSV, TSV) sem reinventar um parser, carregando-os num `DataFrame` do `pandas` — e reconhecer quando a tabela é só de números e vale ir direto a um array do `numpy`
- Extrair informação estruturada de uma página HTML com Beautiful Soup, ler uma tabela HTML com `pd.read_html`, e reconhecer os limites de cada um
- Interpretar respostas de API em JSON, achatar o aninhamento numa tabela com `pd.json_normalize`, e diferenciar uma API autenticada de uma não autenticada
- Explicar por que código que depende de um serviço de terceiros tende a parar de funcionar, e o que fazer diante disso
- Desconfiar da inferência de tipo de um leitor de arquivo — dizer o que ela erra, e corrigi-la com `dtype`, `na_values` e `parse_dates`

## Seções

| Seção | Tópico |
|---|---|
| [6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) | Lendo Arquivos |
| [6.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-raspando-a-web.html) | Raspando a Web |
| [6.3](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/03-usando-apis.html) | Usando APIs |
| [6.4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/04-exemplo-apis-do-twitter.html) | Exemplo: As APIs do Twitter |

## Lendo Arquivos

> **📌 Nota**
>
> Esta seção corresponde a *Reading Files*, do capítulo 9 de Grus (2019).

O primeiro degrau da escada é o mais firme: o seu próprio disco. Um arquivo está onde você o deixou, no formato que você conhece, e não deixa de existir porque um serviço de terceiros mudou de ideia. Ler e escrever arquivos diretamente no código é simples em Python, e é daqui que parte quase todo trabalho com dado.

Os arquivos que aparecem abaixo (`email_addresses.txt`, `tab_delimited_stock_prices.txt` etc.) não existem de antemão: cada um é escrito na hora, num diretório temporário que é apagado no fim da seção. A única exceção é `dados/stocks.csv`, que já vem no repositório e entra duas vezes na segunda metade da seção, quando o arquivo deixa de ser de brinquedo — uma vez lido para uma tabela, outra para um array.

In [ ]:
import tempfile
from pathlib import Path

pasta = Path(tempfile.mkdtemp(prefix="cap06-"))
pasta

### Fundamentos de arquivos de texto

O primeiro passo para trabalhar com um arquivo de texto é obter um *objeto arquivo* com `open`:

```python
# 'r' significa somente leitura; é o padrão se você não especificar nada
file_for_reading = open('reading_file.txt', 'r')
file_for_reading2 = open('reading_file.txt')

# 'w' é escrita -- destrói o arquivo se ele já existir!
file_for_writing = open('writing_file.txt', 'w')

# 'a' é anexação -- para adicionar ao final do arquivo
file_for_appending = open('appending_file.txt', 'a')

# não esqueça de fechar os arquivos quando terminar
file_for_writing.close()
```

Como é fácil esquecer de fechar um arquivo, o jeito certo é sempre abri-lo dentro de um bloco `with`, que fecha automaticamente ao final:

In [ ]:
with open(pasta / 'exemplo.txt', 'w') as f:
    f.write("primeira linha\nsegunda linha\n")

with open(pasta / 'exemplo.txt') as f:
    conteudo = f.read()

conteudo

`with` é um **gerenciador de contexto** — a mesma garantia de limpeza que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) já cobriu para exceções, aplicada a arquivos.

Se você precisa ler um arquivo de texto inteiro, basta iterar sobre as linhas com `for`:

In [ ]:
import re

with open(pasta / 'exemplo2.txt', 'w') as f:
    f.write("# título\nlinha normal\n# outro comentário\nmais uma linha\n")

starts_with_hash = 0
with open(pasta / 'exemplo2.txt') as f:
    for line in f:                    # olha cada linha do arquivo
        if re.match("^#", line):      # usa uma regex para ver se começa com '#'
            starts_with_hash += 1     # se começa, soma 1 à contagem

starts_with_hash

O `re.match` ancora no começo da string — a mesma distinção entre `match` e `search` que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/06-ferramentas-e-tipos.html) já cobriu. E iterar assim sobre `f` usa o padrão de gerador que o [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/05-testes-classes-e-geradores.html) apresentou: cada `for line in f` produz uma linha por vez, sem carregar o arquivo inteiro na memória — o que importa quando o arquivo tem gigabytes.

Toda linha lida desse jeito termina com um caractere de nova linha, então normalmente você vai querer aplicar `.strip()` nela antes de fazer qualquer coisa. Um exemplo: imagine um arquivo com um endereço de e-mail por linha, e você quer montar um histograma dos domínios. A regra para extrair um domínio corretamente é sutil — ela erra em endereços como `joel@m.datasciencester.com` —, mas uma primeira aproximação razoável é pegar tudo depois do `@`:

In [ ]:
from collections import Counter

with open(pasta / 'email_addresses.txt', 'w') as f:
    f.write("joelgrus@gmail.com\n")
    f.write("joel@m.datasciencester.com\n")
    f.write("joelgrus@m.datasciencester.com\n")

def get_domain(email_address: str) -> str:
    """Separa por '@' e devolve o último pedaço"""
    return email_address.lower().split("@")[-1]

# alguns testes
assert get_domain('joelgrus@gmail.com') == 'gmail.com'
assert get_domain('joel@m.datasciencester.com') == 'm.datasciencester.com'

with open(pasta / 'email_addresses.txt', 'r') as f:
    domain_counts = Counter(get_domain(line.strip())
                            for line in f
                            if "@" in line)

domain_counts

### Arquivos delimitados: o que existe por baixo

O arquivo de e-mails tinha um endereço por linha. Mais frequentemente você vai lidar com arquivos que têm vários campos por linha, quase sempre separados por vírgula ou por tabulação — o formato em que o VP de Receita da DataSciencester manda a cotação diária de ações de outro time, por exemplo, sempre que precisa que alguém cruze aquilo com os números da casa.

Em princípio dá para processar esses arquivos lendo linha a linha e cortando pelo separador, mas isso complica rápido quando algum campo contém o próprio caractere separador, ou uma quebra de linha. **Por essa razão, nunca faça o parsing na mão.** O caminho que você vai usar na prática é o `pandas`, assunto do próximo bloco; esta passagem mostra antes o módulo `csv` da biblioteca padrão, que faz à mão o mesmo trabalho — vale conhecer o mecanismo uma vez, para saber o que o `read_csv` está decidindo por você.

Se o arquivo não tem cabeçalho, `csv.reader` itera sobre as linhas já divididas em listas:

In [ ]:
import csv

with open(pasta / 'tab_delimited_stock_prices.txt', 'w') as f:
    f.write("6/20/2014\tAAPL\t90.91\n")
    f.write("6/20/2014\tMSFT\t41.68\n")
    f.write("6/20/2014\tFB\t64.5\n")
    f.write("6/19/2014\tAAPL\t91.86\n")
    f.write("6/19/2014\tMSFT\t41.51\n")
    f.write("6/19/2014\tFB\t64.34\n")

def process(date: str, symbol: str, closing_price: float) -> None:
    # imagine que esta função faz alguma coisa de verdade
    assert closing_price > 0.0

with open(pasta / 'tab_delimited_stock_prices.txt') as f:
    tab_reader = csv.reader(f, delimiter='\t')
    for row in tab_reader:
        date = row[0]
        symbol = row[1]
        closing_price = float(row[2])
        process(date, symbol, closing_price)

"processado sem erro"

O `dados/iris.data` do [Capítulo 9](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap09/02-exemplo-o-dataset-iris.html) tem o mesmo tipo de linha — quatro medidas e um rótulo de texto no fim —, a mesma mistura de número e texto, só que os campos são medidas de flor em vez de preços de ação. É uma mistura que o array do `numpy` não aceita e que tanto o `csv.reader` quanto a ferramenta do próximo bloco resolvem sem esforço.

> **⚠️ Atenção — Linhas malformadas não avisam sozinhas**
>
> `float(row[2])` confia que o terceiro campo é sempre um número. Dados reais quebram essa promessa o tempo todo — um valor ausente, um "N/D", um campo cortado pela metade. Sem tratamento, uma única linha ruim derruba o processamento do arquivo inteiro.
>
> O [Capítulo 2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap02/02-funcoes-strings-excecoes.html) já apresentou o padrão: capture a exceção específica que sabe tratar, não um `except:` pelado. Aplicado aqui, isso significa pular a linha malformada e seguir para a próxima, em vez de estourar:

In [ ]:
with open(pasta / 'precos_com_erro.txt', 'w') as f:
    f.write("6/20/2014\tAAPL\t90.91\n")
    f.write("6/20/2014\tMSFT\tN/D\n")   # preço malformado
    f.write("6/20/2014\tFB\t64.5\n")

precos_validos = []
linhas_puladas = 0

with open(pasta / 'precos_com_erro.txt') as f:
    tab_reader = csv.reader(f, delimiter='\t')
    for row in tab_reader:
        date, symbol, raw_price = row
        try:
            closing_price = float(raw_price)
        except ValueError:
            linhas_puladas += 1
            continue
        precos_validos.append((date, symbol, closing_price))

precos_validos, linhas_puladas

> Duas linhas processadas, uma pulada — e o programa não precisou saber de antemão que a linha do MSFT estava quebrada.

Quando o arquivo tem cabeçalho, o módulo `csv` oferece ainda o `csv.DictReader`, que devolve cada linha como um `dict` com o cabeçalho nas chaves. Não vamos usá-lo: é exatamente o serviço que o `read_csv` presta no próximo bloco, e ele o presta melhor — o `DictReader` devolve strings, e a conversão de cada campo continuaria sendo sua.

Para escrever dados delimitados, use `csv.writer`:

In [ ]:
todays_prices = {'AAPL': 90.91, 'MSFT': 41.68, 'FB': 64.5}

with open(pasta / 'comma_delimited_stock_prices.txt', 'w') as f:
    csv_writer = csv.writer(f, delimiter=',')
    for stock, price in todays_prices.items():
        csv_writer.writerow([stock, price])

(pasta / 'comma_delimited_stock_prices.txt').read_text()

`csv.writer` faz a coisa certa se os campos em si contiverem vírgulas. Um escritor feito à mão provavelmente não faria:

In [ ]:
resultados = [["test1", "success", "Monday"],
              ["test2", "success, kind of", "Tuesday"],
              ["test3", "failure, kind of", "Wednesday"],
              ["test4", "failure, utter", "Thursday"]]

# não faça isto!
with open(pasta / 'bad_csv.txt', 'w') as f:
    for row in resultados:
        f.write(",".join(map(str, row)))  # pode ter vírgulas demais aqui dentro!
        f.write("\n")                     # a linha também pode ter quebras de linha!

print((pasta / 'bad_csv.txt').read_text())

> **⚠️ Atenção**
>
> Repare no que aconteceu: `"success, kind of"` tinha uma vírgula própria, e o arquivo resultante não tem como distinguir essa vírgula das que separam campos. Abrindo `bad_csv.txt` com qualquer leitor de CSV de verdade, a segunda linha vira **quatro** campos em vez de três — e não há mensagem de erro nenhuma avisando disso. É exatamente o tipo de bug silencioso que `csv.writer` existe para evitar. O `DataFrame` do próximo bloco escreve com `df.to_csv(...)`, e segue as mesmas regras de aspas do `csv.writer` — pelo motivo simples de ser o mesmo problema.

### O DataFrame

O `csv.reader` devolveu listas de strings, e todo o resto — converter, nomear campo, juntar linhas — ficou por sua conta. Fora deste livro quase ninguém faz isso: a ferramenta padrão de quem trabalha com dado tabular é a biblioteca **`pandas`**, e o objeto que ela oferece é o **`DataFrame`** — uma tabela com colunas **nomeadas**, cada uma com o **seu próprio tipo**. É a diferença que importa em relação ao array que o próximo bloco apresenta: num array, ou tudo é número, ou nada é; num `DataFrame`, uma coluna de datas, uma de texto e seis de número convivem lado a lado, cada uma sabendo o que é.

A partir daqui e até o fim do livro, **ler, limpar, agrupar e apresentar dado é trabalho do `pandas`**. Calcular o modelo continua sendo trabalho seu, com o `numpy` de calculadora. Este bloco é a apresentação da mesa de trabalho.

In [ ]:
import pandas as pd

precos = pd.read_csv(pasta / 'tab_delimited_stock_prices.txt',
                     sep='\t', header=None,
                     names=["data", "simbolo", "fechamento"],
                     parse_dates=["data"])
precos

Uma chamada no lugar do `with`, do `csv.reader`, do laço e do `float(row[2])`. Os três argumentos que não são o caminho do arquivo dizem exatamente o que o laço dizia: `sep='\t'` é o `delimiter`, `header=None` avisa que a primeira linha já é dado, `names` batiza as colunas — e `parse_dates` faz o que o laço **não** fazia, transformando `6/20/2014` numa data de verdade em vez de deixá-la como texto.

In [ ]:
precos.dtypes

Três colunas, três tipos. `datetime64[ns]` é uma data — dá para subtrair duas, extrair o dia da semana, ordenar cronologicamente; `float64` é número; `object` é o tipo que o `pandas` usa para texto (e para qualquer coisa que ele não conseguiu classificar melhor, o que vai ser importante mais adiante nesta seção). **`dtypes` é a primeira coisa a olhar depois de ler um arquivo**, e é a que mais evita descobrir três capítulos adiante que a coluna de preços era texto.

#### Um arquivo de verdade

O mesmo leitor escala sem mudar de forma. `dados/stocks.csv` traz 23.105 cotações diárias de quatro ações, com cabeçalho e oito colunas:

```
Symbol,Date,Open,High,Low,Close,Adj Close,Volume
AAPL,1980-12-12,0.513393,0.515625,0.513393,0.513393,0.023106,117258400
```

Como o arquivo tem cabeçalho e é separado por vírgula — o padrão —, sobra um argumento só:

In [ ]:
acoes = pd.read_csv("dados/stocks.csv", parse_dates=["Date"])

print(acoes.shape)
acoes.head()

`shape` é a forma da tabela: 23.105 linhas por 8 colunas. `head()` mostra as cinco primeiras — e repare no que o `csv.reader` teria devolvido aqui: 23.105 listas de oito strings, incluindo a de `Symbol`, a de `Date` e as seis de número, todas indistinguíveis entre si.

In [ ]:
acoes.dtypes

Oito colunas, quatro tipos, e nenhuma linha sua para chegar até aí. `Symbol` é texto, `Date` é data porque você pediu, as cinco colunas de preço são `float64` e `Volume` é `int64` — inteiro, porque no arquivo ele é inteiro e ninguém precisou fingir o contrário. Guarde essa última: o `np.loadtxt` do próximo bloco vai transformar o volume em `float64`, porque um array só tem um tipo.

In [ ]:
acoes[["Close", "Volume"]].describe()

Contagem, média, desvio, mínimo, quartis e máximo, para cada coluna pedida, numa chamada. A coluna `Volume` sai em notação científica porque seus valores são grandes demais para caberem confortavelmente na largura da tabela: `2.310500e+04` é $2{,}3105 \times 10^{4}$, as mesmas 23.105 linhas de sempre, e `1.855410e+09` é o maior volume diário do arquivo, um bilhão e oitocentos e cinquenta e cinco milhões de ações. Não há nada de novo aqui em matéria de estatística — é o resumo de sempre —, e é justamente por isso que ele deve ser a segunda coisa a rodar: um mínimo negativo numa coluna de preço, ou um `count` menor que o número de linhas, aparece nesta tabela antes de virar um problema no modelo. Repare que a mediana do fechamento (26,07) é muito menor que a média (95,33): a distribuição é torta, e são quase quarenta anos de preços de empresas que cresceram muito.

#### Escolher uma coluna, escolher algumas linhas

Uma coluna se escolhe pelo nome, entre colchetes, e o que sai é uma **`Series`** — uma coluna com o seu tipo e o seu índice. Uma lista de nomes devolve um `DataFrame` com essas colunas. E linhas se escolhem por uma condição, exatamente como no `if` de uma compreensão de lista, só que a condição é avaliada para as 23.105 de uma vez:

In [ ]:
print(acoes["Close"].head(3))

acoes["Symbol"].value_counts()

O rodapé `Name: Close, dtype: float64` é a assinatura de uma `Series`: um nome só e um tipo só, contra as oito colunas e os quatro tipos que o `DataFrame` mostrou acima. E `value_counts()` conta quantas vezes cada valor aparece na coluna — o `Counter` dos domínios de e-mail lá do começo da seção, embutido. As quatro ações estão nomeadas e contadas: 9.584 pregões da AAPL, 8.259 da MSFT, 3.607 do GOOG e 1.655 do FB, que somam as 23.105 linhas. E para ficar só com uma delas:

In [ ]:
msft = acoes[acoes["Symbol"] == "MSFT"]

print(msft.shape)
msft[["Date", "Close"]].tail(3)

`acoes["Symbol"] == "MSFT"` não devolve `True` ou `False`: devolve uma coluna de booleanos, um por linha, e usar essa coluna como índice fica com as linhas em que ela é `True`. É o mesmo mecanismo que a [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) vai apresentar para arrays, com o nome de *máscara booleana*. Repare no índice das últimas três linhas — 17840, 17841, 17842: ele veio da tabela original e **não** foi renumerado. As 8.259 linhas da MSFT vêm depois das 9.584 da AAPL no arquivo, então a primeira linha de `msft` é a de rótulo 9584, e a última é a 17842. O `DataFrame` guarda de onde cada linha veio, o que é útil e é uma das poucas coisas capazes de surpreender quem chega do array.

#### A inferência de tipo é uma heurística, não uma garantia

Nada do que você leu até aqui declarou um tipo, e é isso que torna o `read_csv` tão confortável: ele **adivinha**. Adivinhar bem na maioria das vezes é diferente de acertar sempre, e as duas maneiras de errar aparecem juntas no cadastro abaixo — um CEP que começa com zero, e um preço ausente marcado com `N/D`, o mesmo marcador do `precos_com_erro.txt` que o callout *Linhas malformadas não avisam sozinhas* usou mais atrás nesta seção:

In [ ]:
with open(pasta / 'cadastro.csv', 'w') as f:
    f.write("nome,cep,gasto\n")
    f.write("Ana,01310,1250.00\n")
    f.write("Bruno,04567,N/D\n")
    f.write("Carla,70040,980.50\n")

cadastro = pd.read_csv(pasta / 'cadastro.csv')

print(cadastro.dtypes)
cadastro

Dois estragos, nenhum aviso. O CEP `01310` virou o **inteiro** 1310: a coluna era só de dígitos, o `pandas` concluiu que era número, e o zero à esquerda não sobrevive a essa conclusão — endereço nenhum fica em São Paulo com CEP 1310. E a coluna `gasto`, por causa de um único `N/D`, não virou número nenhum: ficou `object`, isto é, três strings. O `pandas` conhece uma lista de marcadores de ausência e converte esses sozinho — `NA`, `N/A`, `n/a`, `null`, `NaN`, campo vazio —, mas `N/D` é abreviação em português e não está nela.

A parte pior é o que acontece com a coluna de texto adiante:

In [ ]:
try:
    print(cadastro["gasto"].mean())
except TypeError as erro:
    print("mean():", erro)

print("sum():", repr(cadastro["gasto"].sum()))

A média reclama — ótimo. A **soma não**: `+` entre strings concatena, então `sum()` devolve `'1250.00N/D980.50'`, uma string, com toda a cara de resultado. É a mesma classe de falha do `bad_csv.txt` e do `nan` que aparece adiante nesta mesma seção: nenhuma exceção, nenhuma mensagem, uma resposta errada.

A correção não é ler diferente — é **declarar** o que o arquivo tem, o que são dois argumentos:

In [ ]:
cadastro = pd.read_csv(pasta / 'cadastro.csv',
                       dtype={"cep": str},      # CEP é código, não número
                       na_values=["N/D"])       # o marcador de ausência deste arquivo

print(cadastro.dtypes)
print(cadastro.isna().sum())
cadastro

`dtype={"cep": str}` diz que aquela coluna é código, não quantidade — a pergunta que decide é se faz sentido somar dois valores dela. `na_values=["N/D"]` acrescenta o marcador deste arquivo à lista que o `pandas` já conhece, e com isso `gasto` vira `float64` com um `NaN` no lugar certo: `cadastro["gasto"].mean()` agora devolve 1115,25, a média dos dois valores que existem.

E `isna().sum()` conta os ausentes por coluna. **Rode-o toda vez, logo depois do `dtypes`**: o dado que falta é a informação que menos se anuncia e mais estraga conta adiante. O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/04-limpeza-e-transformacao.html) volta a esse `NaN` para decidir o que fazer com ele — descartar a linha, voltar à fonte e corrigir, ou seguir sabendo que ele está ali.

### Quando o destino é o modelo: `np.loadtxt`

O `DataFrame` é a mesa de trabalho: é onde o dado é lido, tipado, limpo e conferido. Mas o que este livro vai **calcular** daqui em diante — o gradiente, a distância entre dois pontos, a matriz de coeficientes — não trabalha com colunas nomeadas: trabalha com uma tabela em que tudo é número, do mesmo tipo, na mesma memória. É o *array* que o callout do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/01-vetores.html) mostrou de relance, e ele é a **calculadora** deste livro, do jeito que o `pandas` é a mesa.

Quando o arquivo já é só número, dá para ir direto a ele, sem passar pela mesa: o `numpy` tem um leitor próprio.

O arquivo de tabulações de cima, lido de novo:

In [ ]:
import numpy as np

fechamentos = np.loadtxt(pasta / 'tab_delimited_stock_prices.txt',
                         delimiter='\t', usecols=2)
fechamentos

Uma chamada fez o que o laço com `csv.reader` e `float(...)` fez acima — mas repare no `usecols=2`. Um array guarda valores de **um único tipo**, e as duas primeiras colunas do arquivo são texto; sem o `usecols`, o `np.loadtxt` tentaria converter `6/20/2014` em número e pararia ali, com um `ValueError`. Ele não substitui o `csv`: é o leitor certo para a parte numérica de um arquivo, e só para ela.

Sobre o `dados/stocks.csv` que o bloco anterior leu, o mesmo leitor precisa de dois argumentos a mais, e é instrutivo ver por quê: `skiprows=1` pula o cabeçalho, que o `read_csv` usou para nomear as colunas, e `usecols=range(2, 8)` descarta `Symbol` e `Date`, que o `read_csv` guardou como texto e como data:

In [ ]:
precos_stocks = np.loadtxt("dados/stocks.csv", delimiter=",",
                           skiprows=1, usecols=range(2, 8))
precos_stocks.shape, precos_stocks.dtype

`shape` diz o que chegou: 23.105 linhas por 6 colunas, uma matriz no sentido do Capítulo 4 — e `dtype` diz que tudo virou `float64`, inclusive o volume, que no arquivo era inteiro.

Os dois leitores chegam ao mesmo lugar. O `DataFrame` já lido tem as mesmas seis colunas de número; pedi-las como array é um método:

In [ ]:
colunas = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
X = acoes[colunas].to_numpy()

X.shape, X.dtype, np.array_equal(X, precos_stocks)

Mesma forma, mesmo tipo, e `np.array_equal` confirma: os mesmos 138.630 números, lidos por dois caminhos. Repare no que `.to_numpy()` teve de fazer com a coluna `Volume`, que no `DataFrame` era `int64`: num array só existe um tipo, e ele virou `float64` junto com as outras cinco.

A escolha entre os dois caminhos não é de gosto. Quando o arquivo é só número e o destino é uma conta, o `np.loadtxt` vai direto. Quando o arquivo tem uma data, um rótulo, um valor faltando ou uma coluna que precisa ser limpa antes — isto é, quase sempre —, o dado entra pelo `DataFrame` e sai para o array **na hora em que o modelo começa**, com um `.to_numpy()` como o de cima. O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html) dá nome a essa passagem e a repete em todo capítulo que ajusta um modelo.

A fatia `precos_stocks[:, 3]` pega a coluna do fechamento inteira — o `get_column` do [Capítulo 4](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap04/02-matrizes.html), sem percorrer linha nenhuma:

In [ ]:
abertura = precos_stocks[:, 0]      # Open
fechamento = precos_stocks[:, 3]    # Close

variacao_pct = (fechamento - abertura) / abertura * 100

f"maior alta num pregão: {variacao_pct.max():.2f}%; maior queda: {variacao_pct.min():.2f}%"

Não há `for` em lugar nenhum: a subtração, a divisão e o `* 100` valem para as 23.105 linhas de uma vez. É o `subtract` e o `scalar_multiply` que você escreveu no Capítulo 4, agora sobre arrays — e a divisão elemento a elemento, que o Capítulo 4 nem chegou a escrever, segue a mesma regra. É isso, mais do que a leitura do arquivo, que o `numpy` traz para o resto do livro.

Falta o caso que esta seção já ensinou a temer. O `precos_com_erro.txt` de cima tem um `N/D` no lugar de um preço; o `np.loadtxt` se recusa a lê-lo, e o irmão mais tolerante, `np.genfromtxt`, lê — colocando `nan` (*not a number*) onde não conseguiu converter:

In [ ]:
try:
    np.loadtxt(pasta / 'precos_com_erro.txt', delimiter='\t', usecols=2)
except ValueError as erro:
    print(erro)

precos_sujos = np.genfromtxt(pasta / 'precos_com_erro.txt', delimiter='\t', usecols=2)
precos_sujos, precos_sujos.mean(), np.isnan(precos_sujos)

O `nan` não avisa: a média dos três valores sai `nan`, sem erro e sem mensagem, e qualquer conta que passe por ele devolve `nan` também. É a mesma classe de falha silenciosa do callout *Linhas malformadas não avisam sozinhas* — lá o `try/except` era seu; aqui a tolerância é do `genfromtxt`, e o `np.isnan` é o que sobra para descobrir onde o buraco está. Na dúvida, prefira o leitor que reclama: um `ValueError` na leitura custa menos que um `nan` descoberto três capítulos depois. São três leitores e três respostas para o mesmo `N/D`: o `csv` com `try/except` descartou a linha e contou quantas; o `read_csv` sem `na_values` deixou a coluna virar texto; o `genfromtxt` pôs `nan`. Nenhuma das três está errada — errado é não saber qual delas você escolheu.

> **📌 Nota**
>
> Isto foi só o primeiro contato com o array. O que ele é por dentro — `dtype`, fatias, máscaras booleanas, `axis`, *broadcasting* — é o assunto do bloco *De listas a arrays* que abre a [seção 7.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/01-explorando-seus-dados.html), e é lá que o `numpy` passa a ser a calculadora de todos os modelos deste livro. Do `pandas`, o bloco acima já entregou o que o resto do livro usa — `read_csv`, `dtypes`, `describe`, seleção de coluna e de linha, e a desconfiança saudável da inferência de tipo. O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html) acrescenta o que falta para trabalhar de verdade: limpar, agrupar e resumir.

In [ ]:
import shutil
shutil.rmtree(pasta, ignore_errors=True)

> **💡 Dica — Na prática: o que a conveniência esconde**
>
> Esta seção usou três leitores para o mesmo tipo de arquivo, e vale ver o balanço lado a lado:
>
> | leitor | decide o tipo | linha ruim | tipos diferentes na mesma tabela |
> |---|---|---|---|
> | `csv.reader` | **você**, campo a campo | você trata, e sabe que tratou | sim, mas tudo chega como texto |
> | `pd.read_csv` | ele, por heurística | vira `NaN` ou coluna `object`, em silêncio | sim, uma coluna, um tipo |
> | `np.loadtxt` | ele, para um tipo só | `ValueError`, e para de ler | não |
>
> Nenhum é o certo sempre, e o `read_csv` é o padrão fora deste livro porque erra menos vezes no caso comum — não porque não erra. Três coisas que ele faz por você e que só aparecem quando incomodam:
>
> **O índice.** Todo `DataFrame` tem um índice de linha, e ele não é a posição: a primeira linha do `msft` filtrado acima é a linha 0 **dele**, mas o rótulo dela continua sendo 9584, que era a posição no arquivo original — e um `reset_index(drop=True)` é o que renumera. Confundir rótulo com posição é o erro nº 1 de quem chega do array — por isso existem `.loc` (por rótulo) e `.iloc` (por posição), separados de propósito.
>
> **A memória.** O `read_csv` lê o arquivo inteiro de uma vez. `dados/stocks.csv` tem 1,7 MB e isso não é problema; um CSV de 20 GB é. Para esse caso existem `chunksize` (lê em pedaços), `usecols` (só as colunas que interessam — o mesmo nome do `np.loadtxt`, e a mesma ideia) e formatos colunares como o Parquet, que não precisam ser lidos por inteiro para serem consultados.
>
> **O `object`.** Uma coluna que o `pandas` não conseguiu classificar vira `object`, e `object` engole qualquer coisa sem reclamar — foi o que aconteceu com o `gasto` acima. É o tipo mais perigoso de encontrar num `dtypes`, porque é o único que significa "não sei".
>
> A regra que vale para os três leitores é a que o resto deste livro cobra dos modelos: a conta pode ficar com a biblioteca; **a decisão que importa continua sendo uma linha sua** — e num `read_csv` essa linha são os argumentos `dtype`, `na_values` e `parse_dates`, escritos porque você olhou o arquivo, não porque deram erro.

## Raspando a Web

> **📌 Nota**
>
> Esta seção corresponde a *Scraping the Web*, do capítulo 9 de Grus (2019).

Depois do seu próprio disco vem o HTML de outra pessoa: dá para obter dado raspando páginas web diretamente. Buscar a página é a parte fácil; extrair dela informação estruturada e com sentido, bem menos.

### HTML e como interpretá-lo

Páginas na web são escritas em HTML, no qual o texto é (idealmente) marcado em elementos e seus atributos:

```html
<html>
  <head>
    <title>Uma página web</title>
  </head>
  <body>
    <p id="author">Joel Grus</p>
    <p id="subject">Data Science</p>
  </body>
</html>
```

Num mundo perfeito, onde toda página web fosse marcada de forma semântica em nosso benefício, daria para extrair dado com regras do tipo "ache o elemento `<p>` cujo `id` é `subject` e devolva o texto que ele contém." No mundo real, HTML raramente é bem formado, e quase nunca é anotado — então vamos precisar de ajuda para dar sentido a ele.

Para isso usamos a biblioteca **Beautiful Soup**, que constrói uma árvore a partir dos elementos de uma página e oferece uma interface simples para acessá-los. E usamos a biblioteca **Requests**, um jeito bem mais agradável de fazer requisições HTTP do que qualquer coisa embutida em Python. O parser HTML embutido do Python não é muito tolerante com marcação malformada, então também usamos o parser `html5lib`.

"Tolerante" aqui tem um sentido preciso. Um navegador, ao abrir uma página com uma tag não fechada ou aninhada errado, não trava — ele segue um **algoritmo de recuperação de erros** especificado (o algoritmo de parsing do padrão HTML5), que decide deterministicamente onde cada tag quebrada deveria ter fechado. O parser embutido do Python (`html.parser`) não implementa esse algoritmo, e o problema não é que ele trave: `BeautifulSoup(texto, "html.parser")` sobre uma tag `<p>` aninhada dentro de outra `<p>` não levanta exceção nenhuma — devolve uma árvore **errada**, em silêncio, sem avisar que a marcação estava quebrada. É a mesma classe de falha que a seção anterior ensinou a temer: nenhuma mensagem de erro, só uma resposta incorreta. O `html5lib` implementa o mesmo algoritmo que os navegadores usam, e por isso interpreta a mesma sopa de tags malformadas do jeito que um navegador interpretaria.

Na prática, o HTML viria de uma requisição — `html = requests.get(url).text` — antes de virar `soup = BeautifulSoup(html, 'html5lib')`. Aqui ele vem de uma cópia salva da mesma página, em `dados/getting-data.html`, para que o exemplo dê o mesmo resultado a cada execução. O `BeautifulSoup` recebe exatamente o mesmo texto nos dois casos: só a origem muda.

In [ ]:
from bs4 import BeautifulSoup

with open('dados/getting-data.html') as f:
    html = f.read()

soup = BeautifulSoup(html, 'html5lib')
soup.h1

A partir daqui, tudo funciona com um punhado de métodos simples. Normalmente vamos trabalhar com objetos `Tag`, que correspondem às tags que estruturam a página. Para achar a primeira tag `<p>` (e seu conteúdo):

In [ ]:
first_paragraph = soup.find('p')       # ou simplesmente soup.p

assert str(soup.find('p')) == '<p id="p1">This is the first paragraph.</p>'
first_paragraph

Dá para obter o conteúdo de texto de uma `Tag` pela propriedade `text`:

In [ ]:
first_paragraph_text = soup.p.text
first_paragraph_words = soup.p.text.split()

assert first_paragraph_words == ['This', 'is', 'the', 'first', 'paragraph.']
first_paragraph_words

E dá para extrair os atributos de uma tag tratando-a como um `dict`:

In [ ]:
first_paragraph_id = soup.p['id']         # levanta KeyError se não houver 'id'
first_paragraph_id2 = soup.p.get('id')    # devolve None se não houver 'id'

assert first_paragraph_id == first_paragraph_id2 == 'p1'
first_paragraph_id

Dá para pegar várias tags de uma vez:

In [ ]:
all_paragraphs = soup.find_all('p')             # ou simplesmente soup('p')
paragraphs_with_ids = [p for p in soup('p') if p.get('id')]

assert len(all_paragraphs) == 2
assert len(paragraphs_with_ids) == 1
len(all_paragraphs), len(paragraphs_with_ids)

Frequentemente você vai querer achar tags com uma `class` específica:

In [ ]:
important_paragraphs = soup('p', {'class': 'important'})
important_paragraphs2 = soup('p', 'important')
important_paragraphs3 = [p for p in soup('p')
                         if 'important' in p.get('class', [])]

assert important_paragraphs == important_paragraphs2 == important_paragraphs3
assert len(important_paragraphs) == 1
important_paragraphs

E dá para combinar esses métodos para lógica mais elaborada. Por exemplo, para achar todo elemento `<span>` que está dentro de um `<div>`:

In [ ]:
# Aviso: vai devolver o mesmo <span> várias vezes se ele estiver
# dentro de vários <div>. Seja mais cuidadoso se for o seu caso.
spans_inside_divs = [span
                     for div in soup('div')     # para cada <div> da página
                     for span in div('span')]   # ache cada <span> dentro dele

assert len(spans_inside_divs) == 3
len(spans_inside_divs)

Só esse punhado de recursos já permite fazer bastante coisa. Se você acabar precisando de algo mais elaborado (ou só estiver curioso), vale consultar a [documentação do Beautiful Soup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/). O dado que interessa raramente vem rotulado como `class="important"`, claro — é preciso inspecionar o HTML fonte com cuidado, entender a lógica de seleção e prestar atenção a casos de borda para garantir que o dado extraído é o correto.

### Exemplo: de olho no Congresso

O VP de Políticas Públicas da DataSciencester está preocupado com uma possível regulação da indústria de ciência de dados, e pede que você quantifique o que o Congresso norte-americano anda dizendo sobre o tema. Em particular, ele quer achar todos os representantes que têm comunicados de imprensa sobre "data".

Ao tempo em que o Grus escreveu o livro, havia uma página com links para os sites de todos os representantes, em `https://www.house.gov/representatives`, e cada link tinha esta forma:

```html
<td>
  <a href="https://jayapal.house.gov">Jayapal, Pramila</a>
</td>
```

A função abaixo — pura lógica sobre uma string de HTML, sem tocar a rede — verifica se algum `<p>` de um texto menciona uma palavra-chave:

In [ ]:
def paragraph_mentions(text: str, keyword: str) -> bool:
    """
    Devolve True se algum <p> dentro do texto menciona {keyword}
    """
    soup = BeautifulSoup(text, 'html5lib')
    paragraphs = [p.get_text() for p in soup('p')]

    return any(keyword.lower() in paragraph.lower()
               for paragraph in paragraphs)

text = """<body><h1>Facebook</h1><p>Twitter</p></body>"""
assert paragraph_mentions(text, "twitter")       # está dentro de um <p>
assert not paragraph_mentions(text, "facebook")  # não está dentro de um <p>

paragraph_mentions(text, "twitter")

O restante do fluxo — coletar os links da página de representantes, filtrar por regex os que terminam em `.house.gov`, visitar cada site, achar a página de comunicados de imprensa, e procurar "data" em cada um — são dezenas a centenas de requisições HTTP encadeadas, todas dependendo de o `house.gov` estar no ar e responder exatamente como respondia quando o capítulo foi escrito:

```python
from bs4 import BeautifulSoup
import requests
import re
from typing import Dict, Set

url = "https://www.house.gov/representatives"
text = requests.get(url).text
soup = BeautifulSoup(text, "html5lib")

all_urls = [a['href']
           for a in soup('a')
           if a.has_attr('href')]

print(len(all_urls))  # 965 para o Grus, muitos demais

# Precisa começar com http:// ou https://
# Precisa terminar com .house.gov ou .house.gov/
regex = r"^https?://.*\.house\.gov/?$"

# Vamos escrever alguns testes!
assert re.match(regex, "http://joel.house.gov")
assert re.match(regex, "https://joel.house.gov")
assert re.match(regex, "http://joel.house.gov/")
assert re.match(regex, "https://joel.house.gov/")
assert not re.match(regex, "joel.house.gov")
assert not re.match(regex, "http://joel.house.com")
assert not re.match(regex, "https://joel.house.gov/biography")

# E aplica
good_urls = [url for url in all_urls if re.match(regex, url)]

print(len(good_urls))  # ainda 862 para o Grus

# Usa um set para eliminar duplicatas
good_urls = list(set(good_urls))

print(len(good_urls))  # só 431 para o Grus — a Câmara tem 435 cadeiras;
                        # sempre sobra alguma vaga vazia ou site fora do ar

# Na maioria dos sites, há um link para comunicados de imprensa
html = requests.get('https://jayapal.house.gov').text
soup = BeautifulSoup(html, 'html5lib')

# Usa um set porque os links podem aparecer mais de uma vez.
links = {a['href'] for a in soup('a') if 'press releases' in a.text.lower()}
print(links)  # {'/media/press-releases'}

press_releases: Dict[str, Set[str]] = {}

for house_url in good_urls:
    html = requests.get(house_url).text
    soup = BeautifulSoup(html, 'html5lib')
    pr_links = {a['href'] for a in soup('a')
               if 'press releases' in a.text.lower()}
    print(f"{house_url}: {pr_links}")
    press_releases[house_url] = pr_links

for house_url, pr_links in press_releases.items():
    for pr_link in pr_links:
        url = f"{house_url}/{pr_link}"
        text = requests.get(url).text

        if paragraph_mentions(text, 'data'):
            print(f"{house_url}")
            break  # termina com este house_url
```

Este é o exemplo mais datado do capítulo, e vale reparar por quê: os números nos comentários (`965`, `862`, `431`) são o resultado de uma execução específica, não uma constante do site. O `house.gov` pode reformular a página, um representante pode trocar de site, e a lista de representantes muda a cada eleição. O que dura é a técnica — coletar links, filtrar com regex, visitar cada página, aplicar `paragraph_mentions` sobre o resultado —, não os números que ela devolve.

> **📌 Nota**
>
> Raspar um site livremente como o exemplo faz é, em geral, indelicado. A maioria dos sites tem um arquivo `robots.txt` que indica com que frequência (e quais caminhos) você tem permissão de raspar — mas, como observa o próprio Grus, tratando-se do Congresso, não há necessidade de ser particularmente educado.

Vale registrar também uma limitação do exemplo: as páginas de "comunicados de imprensa" de verdade costumam ser paginadas, com só 5 ou 10 comunicados por página. O código acima recupera só os comunicados mais recentes de cada parlamentar — uma solução mais completa precisaria iterar sobre as páginas e recuperar o texto de cada comunicado.

### Quando a página já é uma tabela

Repare no que a página de representantes tinha, e que o exemplo acima ignorou: aquele `<td><a href="...">Jayapal, Pramila</a></td>` não era um `<div>` solto — era a célula de uma **tabela**. Boa parte do dado que se raspa está nessa situação: uma tabela HTML, com cabeçalho e linhas, que alguém já organizou em formato tabular e só não publicou como arquivo.

Para esse caso não é preciso percorrer `<tr>` e `<td>` na mão. O `pandas` tem um leitor que faz isso, e devolve direto o `DataFrame` da [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) — uma tabela por `<table>` encontrada na página:

In [ ]:
import io
import pandas as pd

tabela_html = """
<table>
  <tr><th>Representante</th><th>Estado</th><th>Partido</th></tr>
  <tr><td><a href="https://jayapal.house.gov">Jayapal, Pramila</a></td><td>WA</td><td>D</td></tr>
  <tr><td><a href="https://larsen.house.gov">Larsen, Rick</a></td><td>WA</td><td>D</td></tr>
  <tr><td><a href="https://newhouse.house.gov">Newhouse, Dan</a></td><td>WA</td><td>R</td></tr>
</table>
"""

tabelas = pd.read_html(io.StringIO(tabela_html), flavor="bs4")
representantes = tabelas[0]

representantes

Uma linha no lugar do laço aninhado, com o cabeçalho já virando nome de coluna. Dois detalhes da chamada não são decoração:

`io.StringIO(...)` embrulha a string porque o `read_html` espera um arquivo, uma URL ou algo que se comporte como arquivo — passar HTML solto ainda funciona, mas com um aviso de que vai deixar de funcionar. E `flavor="bs4"` escolhe o parser: o padrão do `pandas` é o `lxml`, que não é uma biblioteca deste livro; com `bs4`, o `read_html` roda em cima do **mesmo Beautiful Soup + html5lib** que você usou nesta seção inteira. Não há mágica nova aqui — há a mesma sopa de tags, com a montagem da tabela feita por outra pessoa.

`read_html` devolve uma **lista** de `DataFrame`, uma por `<table>` da página, e é por isso que o `[0]`. Numa página de verdade, com tabela de navegação e tabela de rodapé, escolher qual delas é a que interessa é parte do trabalho.

Só que falta o que mais importava no exemplo do Congresso: o **endereço** de cada site. O `read_html` lê o *texto* das células, e o `href` é um atributo — ele não aparece na tabela acima. É onde o Beautiful Soup continua sendo necessário, e onde os dois se encontram: a raspagem produz as linhas, o `DataFrame` as recebe.

In [ ]:
soup = BeautifulSoup(tabela_html, 'html5lib')

linhas = [{"representante": a.text, "site": a["href"]}
          for a in soup('a') if a.has_attr('href')]

sites = pd.DataFrame(linhas)
sites

Uma lista de `dict`s vira um `DataFrame` direto — cada chave é uma coluna. Daqui em diante o resultado da raspagem se comporta como qualquer tabela: dá para filtrar por estado, contar por partido, cruzar com outra fonte pelo nome, exportar com `to_csv`. É essa a forma em que dado raspado deve terminar, e é por isso que vale insistir: o `read_html` resolve a tabela em uma linha quando o dado **é** o texto das células; quando o que interessa está num atributo, no `href` ou num `data-`, a raspagem continua sendo o trabalho, e o `DataFrame` é onde ela desemboca.

(O `read_html` tem um `extract_links="body"` que também recupera os `href` — mas ele devolve cada célula como uma tupla `(texto, link)`, inclusive as colunas que não têm link nenhum, e desfazer isso costuma dar mais trabalho que as três linhas acima.)

> **💡 Dica — Na prática: além de Beautiful Soup + Requests**
>
> `BeautifulSoup` e `requests` cobrem o caso comum — HTML estático, já presente na resposta HTTP. Duas limitações reais que eles não resolvem:
>
> Primeiro, páginas que montam o conteúdo via JavaScript **depois** de carregadas: `requests.get` devolve o HTML que o servidor mandou, não o que o navegador constrói em cima dele. Para essas, é preciso um navegador de verdade automatizado — ferramentas como **Selenium** ou **Playwright** abrem um navegador (headless ou não), esperam o JavaScript rodar, e só então extraem o HTML resultante.
>
> Segundo, projetos de raspagem maiores — muitas páginas, navegação por links desconhecidos, reexecução periódica — ganham de uma estrutura dedicada como **Scrapy**, que cuida de fila de requisições, novas tentativas, limite de taxa e exportação de dados, em vez de você reimplementar cada uma dessas partes por cima de um `for` com `requests.get`.
>
> Nenhuma das duas muda a lição central: o dado bruto de uma página raspada é HTML mal formado, e alguma coisa — `BeautifulSoup` ou o que quer que você use — vai ter que impor uma estrutura em cima dele antes que a informação vire algo utilizável. E essa estrutura, quase sempre, é uma tabela: a raspagem termina onde a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) começou.

## Usando APIs

> **📌 Nota**
>
> Esta seção corresponde a *Using APIs*, do capítulo 9 de Grus (2019).

Muitos sites e serviços web oferecem **interfaces de programação de aplicações** (APIs), que permitem pedir dado explicitamente, num formato estruturado. Isso poupa o trabalho de raspar — a página pode mudar de layout a qualquer momento, mas uma API bem versionada não muda o formato da resposta sem aviso.

### JSON e XML

Como HTTP é um protocolo para transferir *texto*, o dado que se pede por uma API web precisa ser **serializado** em alguma forma de string. Frequentemente essa serialização usa **JSON** (*JavaScript Object Notation*). Objetos JavaScript se parecem bastante com `dict`s do Python, o que torna sua representação em string fácil de interpretar:

```json
{ "title" : "Data Science Book",
  "author" : "Joel Grus",
  "publicationYear" : 2019,
  "topics" : [ "data", "science", "data science"] }
```

Dá para interpretar JSON usando o módulo `json` do Python. Em particular, vamos usar sua função `loads`, que desserializa uma string representando um objeto JSON, transformando-a num objeto Python:

In [ ]:
import json

serialized = """{ "title" : "Data Science Book",
                  "author" : "Joel Grus",
                  "publicationYear" : 2019,
                  "topics" : [ "data", "science", "data science"] }"""

# interpreta o JSON para criar um dict Python
deserialized = json.loads(serialized)
assert deserialized["publicationYear"] == 2019
assert "data science" in deserialized["topics"]

deserialized

Às vezes um provedor de API só entrega respostas em **XML**:

```xml
<Book>
  <Title>Data Science Book</Title>
  <Author>Joel Grus</Author>
  <PublicationYear>2014</PublicationYear>
  <Topics>
    <Topic>data</Topic>
    <Topic>science</Topic>
    <Topic>data science</Topic>
  </Topics>
</Book>
```

Dá para usar o Beautiful Soup — o mesmo da [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-raspando-a-web.html) — para extrair dado de XML de forma parecida com o que fizemos com HTML; consulte a documentação para os detalhes.

### Usando uma API sem autenticação

A maioria das APIs hoje em dia exige que você se autentique antes de poder usá-las. Não é capricho dos provedores, mas cobra um bom tanto de código repetitivo que atrapalha a exposição do que interessa. Por isso vamos primeiro olhar a **API do GitHub**, com a qual dá para fazer algumas coisas simples sem autenticação — o bastante, por exemplo, para o VP de Engenharia da DataSciencester descobrir em que dias a equipe mais costuma criar repositórios novos.

A busca em si é uma requisição HTTP:

```python
import requests, json

github_user = "joelgrus"
endpoint = f"https://api.github.com/users/{github_user}/repos"

repos = json.loads(requests.get(endpoint).text)
```

O código acima busca os repositórios da conta `joelgrus` — não a do time de engenharia da DataSciencester do parágrafo anterior. Para apontar a busca a outra conta, basta trocar o valor de `github_user`. O que ele devolve é o estado daquela conta **no momento em que a requisição roda**: rodá-lo hoje e daqui a um mês dá duas listas diferentes.

O que vem depois de `repos` existir é Python comum sobre uma lista de `dict`s — nenhuma linha daqui para baixo toca rede. As datas na resposta chegam como strings, por exemplo `"created_at": "2013-07-05T02:02:28Z"`, e Python não vem com um bom parser de datas embutido; a biblioteca `python-dateutil` cobre esse buraco, e dela você provavelmente só vai precisar da função `dateutil.parser.parse`.

Sobre um `repos` inventado, no mesmo formato que a API devolveria — os cinco repositórios (fictícios) do time de engenharia da DataSciencester —, o processamento fica assim:

In [ ]:
from collections import Counter
from dateutil.parser import parse

repos = [
    {"name": "grafo-de-amizades", "language": "Python",
     "created_at": "2024-03-14T09:12:00Z", "pushed_at": "2024-11-02T17:40:00Z",
     "owner": {"login": "datasciencester", "type": "Organization"}},
    {"name": "painel-de-metricas", "language": "JavaScript",
     "created_at": "2024-03-21T14:05:00Z", "pushed_at": "2024-10-15T08:22:00Z",
     "owner": {"login": "datasciencester", "type": "Organization"}},
    {"name": "modelo-de-recomendacao", "language": "Python",
     "created_at": "2024-06-02T11:47:00Z", "pushed_at": "2024-11-20T13:10:00Z",
     "owner": {"login": "datasciencester", "type": "Organization"}},
    {"name": "relatorios-mensais", "language": "R",
     "created_at": "2024-07-19T16:30:00Z", "pushed_at": "2024-09-05T10:00:00Z",
     "owner": {"login": "vp-analytics", "type": "User"}},
    {"name": "api-interna", "language": "Python",
     "created_at": "2024-09-08T08:55:00Z", "pushed_at": "2024-11-25T19:03:00Z",
     "owner": {"login": "datasciencester", "type": "Organization"}},
]

# Em que meses e dias da semana a equipe mais costuma criar repositórios?
dates = [parse(repo["created_at"]) for repo in repos]
month_counts = Counter(date.month for date in dates)
weekday_counts = Counter(date.weekday() for date in dates)  # segunda=0 ... domingo=6

# E em que linguagem estão os repositórios modificados mais recentemente?
last_5_repositories = sorted(repos,
                             key=lambda r: r["pushed_at"],
                             reverse=True)[:5]

last_5_languages = [repo["language"] for repo in last_5_repositories]

month_counts, weekday_counts, last_5_languages

Repare no `owner`: ele não é um valor, é outro `dict` dentro do primeiro. Respostas de API são assim quase sempre — o GitHub devolve o dono, a licença e as permissões de cada repositório como objetos aninhados, e é o formato natural de quem serializa objetos, não tabelas.

Contra a API de verdade, `repos` teria dezenas ou centenas de entradas em vez de cinco, mas a lógica que extrai mês, dia da semana e linguagem é exatamente esta, rodando sobre `dict`s do Python que teriam chegado pela rede em vez de escritos à mão aqui. Com cinco repositórios, `Counter` e `sorted` bastam. O que se faz no trabalho, a partir da dúzia — e sempre que a resposta precisa ser cruzada com outra tabela, filtrada por três critérios ou virar um relatório —, é achatar o aninhamento numa tabela e deixar as perguntas para o `pandas`.

#### Da resposta aninhada para uma tabela

`pd.json_normalize` recebe a lista de `dict`s como ela veio e devolve um `DataFrame`, resolvendo o aninhamento: cada campo de `owner` vira uma coluna própria, com o nome do caminho até ele.

In [ ]:
import pandas as pd

df = pd.json_normalize(repos)
df["created_at"] = pd.to_datetime(df["created_at"])
df["pushed_at"] = pd.to_datetime(df["pushed_at"])

df.dtypes

Seis colunas, e duas delas — `owner.login` e `owner.type` — não existiam como campos: o `json_normalize` as criou a partir do `dict` aninhado, juntando os nomes com um ponto. As datas ainda chegam como texto (JSON não tem tipo de data), e `pd.to_datetime` faz sobre a coluna inteira o que o `dateutil.parser.parse` fez linha a linha acima — o `UTC` no `dtype` é o `Z` do final de `"2024-03-14T09:12:00Z"`, reconhecido e guardado.

As mesmas três perguntas, agora sobre a tabela:

In [ ]:
print(df["created_at"].dt.month.value_counts().sort_index())
print(df["created_at"].dt.day_name().value_counts())

df.sort_values("pushed_at", ascending=False)[["name", "language"]].head(5)

As mesmas respostas do chunk anterior — dois repositórios criados em março, dois num domingo, e as cinco linguagens na ordem do último *push*. O que mudou foi o custo de fazer a **próxima** pergunta. `.dt` abre os componentes de uma coluna de datas (`.month`, `.day_name()`, `.year`, `.hour`), e `day_name()` já responde `"Thursday"` em vez de `3` — em inglês, como todo rótulo que o `pandas` gera —, sem exigir que ninguém lembre que a semana começa no zero. `value_counts()` é o `Counter`, `sort_values()` é o `sorted`, e o índice preservado (4, 2, 0, 1, 3) diz de que repositório veio cada linha. Quem quiser agora saber quantos repositórios cada dono tem escreve `df.groupby("owner.login")["name"].count()` — uma linha, sobre a tabela que já está pronta; `groupby` agrupa as linhas por valor de uma coluna e é o assunto da [seção 7.5](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/05-manipulando-dados.html).

O formato em que o dado chega já é uma decisão que ecoa adiante: JSON quebrado em campos como `language` e `created_at` deixa pronto exatamente o tipo de [atributo](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap08/06-extracao-e-selecao-de-atributos.html) que um modelo consegue usar — uma API que devolvesse só um parágrafo de texto livre sobre cada repositório teria deixado bem menos pronto.

Normalmente, você não vai trabalhar com APIs neste nível baixo de "montar a requisição e interpretar a resposta na mão". Um dos benefícios de usar Python é que alguém provavelmente já escreveu uma biblioteca para praticamente qualquer API que interesse. Quando essas bibliotecas são bem feitas, elas poupam bastante trabalho de descobrir os detalhes mais espinhosos do acesso à API. (Quando não são bem feitas, ou quando ficam desatualizadas em relação à versão real da API que envolvem, elas podem causar dores de cabeça enormes.)

Mesmo assim, eventualmente você vai precisar escrever sua própria biblioteca de acesso a alguma API (ou, mais provavelmente, depurar por que a de outra pessoa não está funcionando) — então vale conhecer alguns desses detalhes.

### Encontrando APIs

Para uma API específica, vale checar se já existe uma biblioteca Python pronta antes de montar as chamadas na mão: sites como Yelp, Instagram e Spotify têm as suas próprias, e o Grus recomendava uma lista de *wrappers* que o [Real Python](https://realpython.com/) mantinha no GitHub — o tipo de link específico que data primeiro num livro, como a seção seguinte está prestes a ilustrar. Quando não existe nenhuma pronta, sempre resta a raspagem — o último recurso do cientista de dados, coberto na [seção anterior](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-raspando-a-web.html).

> **💡 Dica — Na prática: bibliotecas de acesso a APIs**
>
> O padrão `requests.get` + `json.loads` que você acabou de escrever é exatamente o que uma biblioteca de acesso a API encapsula. `PyGithub`, por exemplo, faz para a API do GitHub o que você faria à mão acima, só que devolvendo objetos Python tipados (`repo.created_at` já vem como `datetime`, não como string) em vez de `dict`s crus:
>
> ```python
> from github import Github
>
> g = Github()
> repos = g.get_user("joelgrus").get_repos()
> ```
>
> O que essas bibliotecas escondem é justamente a parte chata: paginação (a maioria das APIs devolve resultados em páginas de 20 ou 100 itens, e alguém precisa pedir a próxima), limite de taxa (a maioria impõe um número máximo de requisições por hora, e passar do limite derruba sua aplicação até o limite resetar) e nova tentativa em caso de falha de rede transitória. Nenhuma dessas preocupações aparece nos exemplos deste capítulo — eles fazem uma chamada só —, mas qualquer coisa que colete dado de uma API em produção precisa lidar com as três. A [próxima seção](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/04-exemplo-apis-do-twitter.html) mostra outra peça desse quebra-cabeça: autenticação.

## Exemplo: As APIs do Twitter

> **📌 Nota**
>
> Esta seção corresponde a *Example: Using the Twitter APIs*, do capítulo 9 de Grus (2019).

Redes sociais como o Twitter são uma fonte valiosa de dado: notícias em tempo real, reação a eventos correntes, links sobre um assunto específico. E, como boa parte do dado interessante do mundo, o acesso a ele é controlado por uma API.

O acesso gratuito a essa API foi descontinuado depois da publicação do livro-texto, e a biblioteca `twython`, usada aqui, ficou sem manutenção: o código a seguir não funciona mais como escrito, e vale pelo fluxo que ele mostra, não pelo serviço que ele acessa.

### Obtendo credenciais

Para usar a API de uma rede social, tipicamente é preciso criar uma conta de desenvolvedor, abrir um aplicativo dentro dela, e receber um par de chaves — uma **chave de consumidor** (*consumer key*) e um **segredo de consumidor** (*consumer secret*), às vezes chamados de *API key* e *API secret key*. Essas chaves identificam a **aplicação**, não o usuário final — é o aplicativo que está pedindo acesso à API, agindo em nome de alguém.

> **⚠️ Atenção — Chave não é coisa para deixar no código**
>
> As chaves de aplicação são, na prática, uma senha: quem as tiver pode fazer chamadas em nome do seu aplicativo, até o limite que a API permitir. Duas práticas comuns para não deixá-las expostas:
>
> - Guardá-las num arquivo separado (por exemplo `credentials.json`) que **não é** versionado no controle de código, lendo-o com `json.loads` quando o programa inicia.
> - Guardá-las em variáveis de ambiente e lê-las com `os.environ.get`, o padrão que o próprio código do Grus usa abaixo.
>
> Nunca as escreva direto no código-fonte, e muito menos em um repositório público — mesmo privado, um commit antigo com a chave continua lá no histórico depois de "removida" de um commit novo.

### Usando o Twython

A parte mais delicada de usar a API do Twitter, escreve o Grus, é a autenticação — e isso vale para a maioria das APIs. Provedores de API querem garantir que você está autorizado a acessar os dados e que não está estourando os limites de uso deles; também querem saber quem está acessando.

Existem, tipicamente, dois níveis de autenticação: um mais simples (**OAuth 2**), que basta para operações de leitura como buscas; e um mais elaborado (**OAuth 1**), necessário para ações que agem em nome de um usuário — postar, ou (o caso que interessa aqui) conectar-se ao fluxo contínuo de dados.

Com as chaves em mãos, a biblioteca `Twython` automatizava boa parte do fluxo OAuth 1:

```python
import os

# Sinta-se à vontade para colocar sua chave e segredo diretamente
CONSUMER_KEY = os.environ.get("TWITTER_CONSUMER_KEY")
CONSUMER_SECRET = os.environ.get("TWITTER_CONSUMER_SECRET")

import webbrowser
from twython import Twython

# Obtém um cliente temporário para buscar uma URL de autenticação
temp_client = Twython(CONSUMER_KEY, CONSUMER_SECRET)
temp_creds = temp_client.get_authentication_tokens()
url = temp_creds['auth_url']

# Agora visite essa URL para autorizar a aplicação e obter um PIN
print(f"visite {url} e pegue o código PIN, colando-o abaixo")
webbrowser.open(url)
PIN_CODE = input("digite o código PIN: ")

# Agora usamos o PIN_CODE para obter os tokens de verdade
auth_client = Twython(CONSUMER_KEY,
                      CONSUMER_SECRET,
                      temp_creds['oauth_token'],
                      temp_creds['oauth_token_secret'])
final_step = auth_client.get_authorized_tokens(PIN_CODE)
ACCESS_TOKEN = final_step['oauth_token']
ACCESS_TOKEN_SECRET = final_step['oauth_token_secret']

# E obtemos uma nova instância de Twython usando-os.
twitter = Twython(CONSUMER_KEY, CONSUMER_SECRET, ACCESS_TOKEN, ACCESS_TOKEN_SECRET)
```

Vale reconhecer a forma desse fluxo, mesmo sem a `twython`: um cliente **temporário** troca a chave da aplicação por uma URL; um humano visita essa URL, autoriza a aplicação de dentro da própria conta, e recebe um PIN de volta; esse PIN é trocado por um **token de acesso** de verdade, que passa a autenticar as chamadas seguintes. É o mesmo desenho geral do OAuth que aparece atrás de "Entrar com o Google" ou "Entrar com o GitHub" em incontáveis sites — a aplicação nunca vê a senha do usuário, só um token que ele concedeu explicitamente e pode revogar depois.

Autenticado, o próximo passo seria fazer buscas:

```python
# Busca tweets contendo a frase "data science"
for status in twitter.search(q='"data science"')["statuses"]:
    user = status["user"]["screen_name"]
    text = status["text"]
    print(f"{user}: {text}\n")
```

### Busca contra fluxo contínuo

A API de busca devolve só um punhado de resultados recentes — útil para uma consulta pontual, pouco útil quando o que se quer é **muito** dado. Para isso existia a **API de streaming**, que conectava a uma amostra do fluxo contínuo de mensagens públicas do Twitter, exigindo o nível mais elaborado de autenticação (OAuth 1, o mesmo obtido acima).

Para acessá-la com `Twython`, era preciso definir uma classe que herdasse de `TwythonStreamer` e sobrescrevesse o método `on_success` — chamado toda vez que um novo dado chegava — e, possivelmente, `on_error`:

```python
from twython import TwythonStreamer

# Acumular dado numa variável global é uma prática pobre,
# mas torna o exemplo bem mais simples
tweets = []

class MyStreamer(TwythonStreamer):
    def on_success(self, data):
        """
        O que fazer quando o Twitter nos manda dado?
        Aqui, data será um dict do Python representando um tweet.
        """
        # Só queremos coletar tweets em inglês
        if data.get('lang') == 'en':
            tweets.append(data)
            print(f"tweet recebido #{len(tweets)}")

        # Para quando já tivermos coletado o suficiente
        if len(tweets) >= 100:
            self.disconnect()

    def on_error(self, status_code, data):
        print(status_code, data)
        self.disconnect()

stream = MyStreamer(CONSUMER_KEY, CONSUMER_SECRET,
                    ACCESS_TOKEN, ACCESS_TOKEN_SECRET)

# começa a consumir mensagens públicas que contenham a palavra-chave 'data'
stream.statuses.filter(track='data')

# se em vez disso quiséssemos consumir uma amostra de *todas* as mensagens públicas
# stream.statuses.sample()
```

`MyStreamer` se conectaria ao fluxo do Twitter e esperaria por dados. A cada mensagem recebida (aqui, um tweet representado como um objeto Python), o Twitter chamaria `on_success`, que a acumularia na lista `tweets` se o idioma fosse inglês, e desconectaria depois de coletar 100.

A partir daí, seria possível analisar o resultado — por exemplo, achando as hashtags mais comuns:

```python
from collections import Counter

top_hashtags = Counter(hashtag['text'].lower()
                       for tweet in tweets
                       for hashtag in tweet["entities"]["hashtags"])

print(top_hashtags.most_common(5))
```

> **📌 Nota**
>
> Cada tweet carrega bastante dado estruturado além do texto — o idioma, as hashtags, menções, geolocalização quando disponível. Num projeto de verdade, você não ia querer depender de uma `list` em memória para guardar os tweets coletados; ia querer salvá-los num arquivo ou banco de dados, para tê-los de forma permanente — a `list tweets` acima desaparece assim que o processo termina.

> **💡 Dica — Na prática: o que sobrevive desta seção**
>
> **Uma dependência externa que morre é o destino normal de todo código que depende de terceiros** — não uma falha de projeto do Grus: o acesso gratuito à API do Twitter que o livro usa foi descontinuado depois da publicação, o produto trocou de nome, e a própria `twython` está sem manutenção.
>
> O que sobrevive não é "copie isto e funcione" — é entender **autenticação de API** como problema geral. Toda API atual que exige login segue alguma variação do mesmo roteiro:
>
> - **Uma chave de aplicação**, obtida uma vez, num painel de desenvolvedor.
> - **Um fluxo de autorização** — hoje quase sempre OAuth 2 com algum tipo de token de curta duração e um *refresh token* para renová-lo, em vez do PIN manual do OAuth 1 usado aqui.
> - **Um limite de taxa** (*rate limit*), que toda aplicação séria respeita e cujo estouro é a causa mais comum de "minha integração parou de funcionar do nada".
> - **Uma distinção entre busca pontual e fluxo contínuo** — a maioria das APIs modernas de redes sociais, quando ainda oferece acesso de terceiros, mantém essa mesma divisão entre "peça um punhado de resultados agora" e "assine um canal e receba dado conforme ele acontece".
>
> O que este livro não pode fazer é lhe dar um exemplo que continue funcionando: qualquer nome de API específico citado aqui corre o risco de ficar desatualizado antes mesmo deste material ser publicado. O que fica é saber reconhecer essas quatro peças — chave, fluxo de autorização, limite de taxa, busca versus fluxo — na documentação de qualquer API nova que você precisar aprender. Os nomes específicos deste capítulo (`Twython`, `CONSUMER_KEY`) morreram junto com o produto que descreviam; a forma do problema, não.

### Adiante

O que você tem, ao final deste capítulo, não é mais um arquivo, uma árvore de tags ou uma resposta de API — é dado na memória, em uma de três formas: listas e `dict`s do Python, como desde o Capítulo 1; um `DataFrame`, quando o dado é uma tabela, que é o caso mais comum e o que a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) apresentou; e um array do `numpy`, quando tudo é número e o destino é uma conta. É para lá que qualquer uma das quatro seções converge, não importa quão frágil ou robusta tenha sido a fonte.

O [Capítulo 7](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap07/index.html) assume que esse trabalho já foi feito, e começa exatamente onde este termina: apresentando os arrays de vez, mostrando como se passa de uma forma à outra, e então explorando, limpando e transformando o dado que agora está em suas mãos.

## Leituras adicionais

A seção "For Further Exploration" do capítulo 9 de Grus (2019) sugere:

- [pandas](https://pandas.pydata.org/) é a biblioteca principal que quem trabalha com dados usa para lidar com — e, em particular, importar — dado, e é a que este capítulo apresenta e passa a usar. A divisão de trabalho do livro é essa, e vale guardá-la: **obter, limpar, juntar, agrupar e apresentar dado é serviço do `pandas`; a conta do modelo é escrita por nós, com o `numpy`.** A [documentação oficial](https://pandas.pydata.org/docs/user_guide/index.html) é boa e o guia de usuário compensa a leitura; `read_csv` sozinho tem mais de cinquenta parâmetros, e três deles — `dtype`, `na_values`, `parse_dates` — resolvem a maior parte dos problemas que a [seção 6.1](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/01-lendo-arquivos.html) mostrou.
- [Scrapy](https://scrapy.org/) é uma biblioteca completa para construir raspadores web complicados, que fazem coisas como seguir links desconhecidos automaticamente — muito além do que `requests` + `BeautifulSoup` fazem na [seção 6.2](https://bragad.github.io/UnDF-Bases5-CienciaDeDados-202602/content/cap06/02-raspando-a-web.html).
- [Kaggle](https://www.kaggle.com/) hospeda uma grande coleção de conjuntos de dados, junto com competições organizadas em torno deles — um jeito de praticar sobre dado real sem primeiro ter que resolver o problema deste capítulo.

## Referências

- **Grus**. *Data Science from Scratch: First Principles with Python*. 2nd ed.. O'Reilly Media. 2019.